Dil modellerinde her $t$ anındaki token'ın kendisinden önceki ($i \le t$) token'ların bilgilerini toplamasını (aggregate) istiyoruz. Bu bilgi toplamanın en yalın hali geçmiş token'ların ortalamasını almaktır (bag of words). Bu ortalamayı:
1. `for` döngüleriyle,
2. Lower triangular matris (`torch.tril`) ile çarparak,
3. `Softmax` fonksiyonu ve maskeleme kullanarak
üç farklı yolla hesaplayacağız. Üçünün aynı sonucu verdiğini `torch.allclose` ile gösterecek ve matris çarpımının neden ağırlıklı ortalama olduğunu detaylıca açıklayacağız.


In [1]:
import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(42)
B, T, C = 4, 8, 2 # Batch = 4, Context = 8, Embedding = 2
x = torch.randn(B, T, C)
print("x.shape (B, T, C):", x.shape)


x.shape (B, T, C): torch.Size([4, 8, 2])


### For Döngüleriyle Geçmişin Ortalamasını Alma

Satır 0: [10.0,  50.0]
Satır 1: [20.0,  70.0]

1. (10.0 + 20.0) / 2 = 15.0
2. (50.0 + 70.0) / 2 = 60.0

(15.0,60.0)

In [11]:
# Hedefimiz: her (b, t) için, 0'dan t'ye kadar olan token'ların ortalaması
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t+1, C) boyutunda geçmiş token'lar
        xbow[b, t] = torch.mean(xprev, 0) # C boyutu boyunca ortalama

print(xbow[0])

tensor([[ 1.9269,  1.4873],
        [ 1.4138, -0.3091],
        [ 1.1687, -0.6176],
        [ 0.8657, -0.8644],
        [ 0.5422, -0.3617],
        [ 0.3864, -0.5354],
        [ 0.2272, -0.5388],
        [ 0.1027, -0.3762]])


### torch.tril ile Çarparak Ağırlıklı Ortalama

In [8]:
# (3,3) sample
torch.manual_seed(42)
a_tril = torch.tril(torch.ones(3, 3))
a = a_tril / torch.sum(a_tril, 1, keepdim=True)# Normalization
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b


print("alt üçgensel matris:")
print(a_tril)
print("\nvğırlık matrisi --> satır toplamları 1:")
print(a)
print("\nveri matrisi:")
print(b)
print("\nher satır bnin geçmiş satırlarındaki ağırlıklarının ortalamasıdır:")
print(c)

alt üçgensel matris:
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

vğırlık matrisi --> satır toplamları 1:
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

veri matrisi:
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])

her satır bnin geçmiş satırlarındaki ağırlıklarının ortalamasıdır:
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [14]:
# Alt üçgen matris oluşturup satır bazında normalize ediyoruz
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)

# Matris çarpımı: (T, T) @ (B, T, C) -> PyTorch batch matmul ile (B, T, C)
xbow2 = wei @ x

print(wei)
print("\n\n",xbow2[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


 tensor([[ 1.9269,  1.4873],
        [ 1.4138, -0.3091],
        [ 1.1687, -0.6176],
        [ 0.8657, -0.8644],
        [ 0.5422, -0.3617],
        [ 0.3864, -0.5354],
        [ 0.2272, -0.5388],
        [ 0.1027, -0.3762]])


### Softmax ile Ağırlıklı Ortalama

In [16]:
# Diğerlerinden farkı: self-attention geldiğinde değerler 1 olmayacak wei / wei.sum() 1'lerle çalışabildiği
# için sorun çıkacaktı bu yüzden softmax kullanmak daha temiz olacaktır

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # Gelecekteki konumlar -inf yapılır
wei = F.softmax(wei, dim=-1) # Softmax(-inf) = 0, satır toplamları = 1

xbow3 = wei @ x

print(wei)
print(xbow3[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[ 1.9269,  1.4873],
        [ 1.4138, -0.3091],
        [ 1.1687, -0.6176],
        [ 0.8657, -0.8644],
        [ 0.5422, -0.3617],
        [ 0.3864, -0.5354],
        [ 0.2272, -0.5388],
        [ 0.1027, -0.3762]])


### torch.allclose

In [17]:
# tril matris çarpımı ve Softmax maskeleme birebir aynıdır:
print("torch.allclose(xbow2, xbow3): ", torch.allclose(xbow2, xbow3))
# float çarpımından dolayı küsurat vardır 
max_diff_1_2 = (xbow - xbow2).abs().max().item()
max_diff_2_3 = (xbow2 - xbow3).abs().max().item()
print(f"Maksimum mutlak fark |xbow - xbow2|: {max_diff_1_2:.10e}")
print(f"Maksimum mutlak fark |xbow2 - xbow3|: {max_diff_2_3:.10e}")

check1 = torch.allclose(xbow, xbow2, atol=1e-7)
check2 = torch.allclose(xbow, xbow3, atol=1e-7)
check3 = torch.allclose(xbow2, xbow3)

print("\nDoğrulama Sonuçları (Floating Point Toleransı ile):")
print("-----------------------------------------------------")
print(f"xbow (Yöntem 1 - For döngüsü) == xbow2 (Yöntem 2 - torch.tril)    : {check1}")
print(f"xbow (Yöntem 1 - For döngüsü) == xbow3 (Yöntem 3 - Softmax)       : {check2}")
print(f"xbow2 (Yöntem 2 - torch.tril) == xbow3 (Yöntem 3 - Softmax)       : {check3}")

torch.allclose(xbow2, xbow3):  True
Maksimum mutlak fark |xbow - xbow2|: 1.1920928955e-07
Maksimum mutlak fark |xbow2 - xbow3|: 0.0000000000e+00

Doğrulama Sonuçları (Floating Point Toleransı ile):
-----------------------------------------------------
xbow (Yöntem 1 - For döngüsü) == xbow2 (Yöntem 2 - torch.tril)    : True
xbow (Yöntem 1 - For döngüsü) == xbow3 (Yöntem 3 - Softmax)       : True
xbow2 (Yöntem 2 - torch.tril) == xbow3 (Yöntem 3 - Softmax)       : True
